# TE04 - Binary Image Post-Processing and Connected Components

This notebook is a **complete training template** in English based on the TE04 statement,
with all tasks documented and no final solutions filled in.

## Teacher Recommendations (from statement)

- Spend time on theory and analysis questions, not only on coding.
- Check the official correction once it is available.
- Keep the OpenCV documentation open while working.

## Reference Links

- OpenCV docs: http://docs.opencv.org/3.2.0/
- OpenCV morphology tutorial: https://docs.opencv.org/3.2.0/d9/d61/tutorial_py_morphological_ops.html
- OpenCV contours tutorial: https://docs.opencv.org/3.2.0/d4/d73/tutorial_py_contours_begin.html
- NumPy docs: https://numpy.org/doc/stable/
- Matplotlib docs: https://matplotlib.org/stable/

## 0. Setup

Expected / optional files for this TE:
- `../data/te04/code_postal.png`
- `../data/te04/passagepieton.jpg`
- `../data/te04/Film_AVI_1.avi`
- `../data/te04/Film_AVI_2.avi`

Fallback already present in the repository:
- `../data/te03/voilier_oies_blanches.jpg`
- `../data/te02/voilier_oies_blanches.jpg`
- `../data/te01/voilier_oies_blanches.jpg`

The setup below creates the output folder and helps you locate available files without failing immediately when optional TE04 data is missing.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('../data/te04')
OUT_DIR = Path('../outputs/te04')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def find_first_existing(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return Path('__missing__')


def to_rgb(image_bgr):
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def show_gray(image, title='', figsize=(6, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_color(image_bgr, title='', figsize=(6, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(to_rgb(image_bgr))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def otsu_small_foreground(image):
    threshold, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if (binary > 0).mean() > 0.5:
        binary = cv2.bitwise_not(binary)
    return threshold, binary


def show_grid(images, titles, *, cols=3, cmap='gray', figsize=(12, 8)):
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for idx, (image, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            plt.imshow(image, cmap=cmap)
        else:
            plt.imshow(to_rgb(image))
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def draw_component_overlay(image_bgr, binary_mask):
    contours, hierarchy = cv2.findContours(binary_mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    overlay = image_bgr.copy()
    if hierarchy is None:
        return overlay, 0, 0

    hierarchy = hierarchy[0]
    external_count = 0
    internal_count = 0
    for idx, contour in enumerate(contours):
        if hierarchy[idx][3] == -1:
            cv2.drawContours(overlay, contours, idx, (255, 255, 0), cv2.FILLED)
            external_count += 1
        else:
            internal_count += 1

    for idx, contour in enumerate(contours):
        if hierarchy[idx][3] == -1:
            cv2.drawContours(overlay, contours, idx, (0, 255, 255), 2)

    return overlay, external_count, internal_count


def open_first_video_source(candidates):
    for candidate in candidates:
        source = str(candidate) if isinstance(candidate, Path) else candidate
        cap = cv2.VideoCapture(source)
        if cap.isOpened():
            return cap, source
        cap.release()
    return cv2.VideoCapture(), 'unavailable'


def run_te04_segmentation(candidates, out_dir, record_name='te04_segmented.avi'):
    cap, source = open_first_video_source(candidates)
    if not cap.isOpened():
        print('No webcam or AVI source is available.')
        return

    writer = cv2.VideoWriter()
    snapshot_idx = 1
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    record_path = out_dir / record_name

    try:
        print(f'Using source: {source}')
        while True:
            ok, frame = cap.read()
            if not ok:
                print('Video stream ended.')
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            _, binary = otsu_small_foreground(gray)
            binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
            overlay, _, _ = draw_component_overlay(frame, binary)

            if not writer.isOpened():
                height, width = overlay.shape[:2]
                fourcc = cv2.VideoWriter_fourcc(*'XVID')
                writer.open(str(record_path), fourcc, 20.0, (width, height), isColor=True)

            cv2.imshow('TE04 Segmentation', overlay)
            if writer.isOpened():
                writer.write(overlay)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            if key == ord('s'):
                snapshot_path = out_dir / f'fra{snapshot_idx}.png'
                cv2.imwrite(str(snapshot_path), overlay)
                print(f'Saved {snapshot_path.name}')
                snapshot_idx += 1
    finally:
        cap.release()
        if writer.isOpened():
            writer.release()
        cv2.destroyAllWindows()


voilier_path = find_first_existing([
    DATA_DIR / 'voilier_oies_blanches.jpg',
    Path('../data/te03/voilier_oies_blanches.jpg'),
    Path('../data/te02/voilier_oies_blanches.jpg'),
    Path('../data/te01/voilier_oies_blanches.jpg'),
])

available_files = {
    'code_postal': DATA_DIR / 'code_postal.png',
    'passage_pieton': DATA_DIR / 'passagepieton.jpg',
    'avi_1': DATA_DIR / 'Film_AVI_1.avi',
    'avi_2': DATA_DIR / 'Film_AVI_2.avi',
    'voilier': voilier_path if voilier_path.exists() else 'not found',
}

for label, path in available_files.items():
    print(f'{label:16s}: {path}')


## 1. Binary Image Post-Processing with Mathematical Morphology

Goal from the statement:
- evaluate the effect of erosion, dilation, opening, closing, and opening-closing on a binary image
- start with a rectangular `3 x 3` structuring element
- compare what you observe with the expected theoretical behavior

### 1.1 Binarization of a source image

Tasks:
1. Choose a binary-processing source image, for example `code_postal.png` if available.
2. Otherwise, reuse `voilier_oies_blanches.jpg` and convert it to grayscale first.
3. Produce a binary image with values in `{0, 255}`.
4. Display both the grayscale image and the binary result.

In [ ]:
binary_source = find_first_existing([
    DATA_DIR / 'code_postal.png',
    voilier_path,
])
if not binary_source.exists():
    raise FileNotFoundError('No binary-processing source image is available for TE04.')

print(f'Selected source: {binary_source}')
img_gray = cv2.imread(str(binary_source), cv2.IMREAD_GRAYSCALE)
if img_gray is None:
    raise FileNotFoundError(f'Unable to read {binary_source}')

initial_threshold, img_binary = otsu_small_foreground(img_gray)
show_grid(
    [img_gray, img_binary],
    [f'Grayscale source ({binary_source.name})', f'Binary source (Otsu T={initial_threshold:.0f})'],
    cols=2,
    figsize=(10, 4),
)


### 1.2 Erosion, dilation, opening, closing, opening-closing

Use the binary image from 1.1 and a rectangular `3 x 3` structuring element.

Tasks:
- erode the binary image
- dilate the binary image
- apply opening
- apply closing
- apply opening followed by closing
- compare all results visually

In [ ]:
kernel_3x3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
print(kernel_3x3)

img_eroded = cv2.erode(img_binary, kernel_3x3, iterations=1)
img_dilated = cv2.dilate(img_binary, kernel_3x3, iterations=1)
img_opened = cv2.dilate(img_eroded, kernel_3x3, iterations=1)
img_closed = cv2.erode(img_dilated, kernel_3x3, iterations=1)
img_open_close = cv2.morphologyEx(img_opened, cv2.MORPH_CLOSE, kernel_3x3)

show_grid(
    [img_binary, img_eroded, img_dilated, img_opened, img_closed, img_open_close],
    ['Binary', 'Eroded', 'Dilated', 'Opened', 'Closed', 'Open then close'],
    cols=3,
    figsize=(12, 8),
)


### 1.3 Interpretation answers

- The observed results match the theory: erosion shrinks the white foreground, while dilation expands it.
- Thin structures and isolated blobs disappear first under erosion because the structuring element no longer fits inside them.
- Dilation and especially closing reduce small gaps and holes by reconnecting nearby foreground pixels.
- Opening-closing is useful here because it removes small noise first and then reconnects cleaner structures.


### 1.4 Change the structuring element

Tasks:
1. Repeat the experiment with different sizes such as `5 x 5` and `7 x 7`.
2. Compare at least three shapes: rectangle, ellipse, cross.
3. Comment on the sensitivity of the result to the structuring element.

In [ ]:
kernels = {
    'rect_5x5': cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
    'ellipse_5x5': cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    'cross_5x5': cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5)),
    'rect_7x7': cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)),
}

opened_by_kernel = [cv2.morphologyEx(img_binary, cv2.MORPH_OPEN, kernel) for kernel in kernels.values()]
closed_by_kernel = [cv2.morphologyEx(img_binary, cv2.MORPH_CLOSE, kernel) for kernel in kernels.values()]

show_grid(
    opened_by_kernel + closed_by_kernel,
    [f'Open {name}' for name in kernels] + [f'Close {name}' for name in kernels],
    cols=4,
    figsize=(14, 8),
)


### 1.5 Re-implement with `cv2.morphologyEx`

Task:
- reproduce the previous processing pipeline using `cv2.morphologyEx` with the appropriate flags
- verify that the results match the versions built from `cv2.erode` and `cv2.dilate`

In [ ]:
img_opened_ex = cv2.morphologyEx(img_binary, cv2.MORPH_OPEN, kernel_3x3)
img_closed_ex = cv2.morphologyEx(img_binary, cv2.MORPH_CLOSE, kernel_3x3)
img_gradient_ex = cv2.morphologyEx(img_binary, cv2.MORPH_GRADIENT, kernel_3x3)

print(f'Open equality check: {np.array_equal(img_opened, img_opened_ex)}')
print(f'Close equality check: {np.array_equal(img_closed, img_closed_ex)}')

show_grid(
    [img_opened_ex, img_closed_ex, img_gradient_ex],
    ['morphologyEx OPEN', 'morphologyEx CLOSE', 'morphologyEx GRADIENT'],
    cols=3,
    figsize=(12, 4),
)


## 2. Extraction and Display of Connected Components

Reminder from the statement:
- thresholding / binarization is only the first part of segmentation
- connected-component labeling is the second part
- the target binary classes here are background (`0`) and birds (`1` or `255`)

### 2.1 Binarize `voilier_oies_blanches.jpg` into background / birds

Tasks:
1. Load `voilier_oies_blanches.jpg` in grayscale.
2. Find a threshold that separates the sky / background from the birds.
3. Produce a clean binary image where birds are foreground.
4. If necessary, add a light morphology post-processing step before labeling.

In [ ]:
if not voilier_path.exists():
    raise FileNotFoundError('No voilier image is available for connected-component analysis.')

print(f'Voilier path: {voilier_path}')
voilier_gray = cv2.imread(str(voilier_path), cv2.IMREAD_GRAYSCALE)
if voilier_gray is None:
    raise FileNotFoundError(f'Unable to read {voilier_path}')

birds_binary = cv2.threshold(voilier_gray, 130, 255, cv2.THRESH_BINARY_INV)[1]
birds_binary_clean = cv2.morphologyEx(birds_binary, cv2.MORPH_OPEN, kernel_3x3)

show_grid(
    [voilier_gray, birds_binary, birds_binary_clean],
    ['voilier grayscale', 'Thresholded birds', 'Opened bird mask'],
    cols=3,
    figsize=(12, 4),
)


### 2.2 Label connected components

Tasks:
- use `cv2.connectedComponents` or `cv2.connectedComponentsWithStats`
- optionally compare with `scipy.ndimage.label` if SciPy is available in your environment
- visualize the label image with a colormap
- inspect how many connected components are detected

In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(birds_binary_clean, connectivity=8)
component_areas = stats[1:, cv2.CC_STAT_AREA]

plt.figure(figsize=(6, 5))
plt.imshow(labels, cmap='nipy_spectral')
plt.title('Connected components label map')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Foreground components (OpenCV, 8-connectivity): {num_labels - 1}')
print(f'Largest component areas: {np.sort(component_areas)[-10:][::-1].tolist()}')
print(f'First three centroids: {centroids[1:4].round(2).tolist()}')

try:
    from scipy import ndimage

    scipy_labels, scipy_count = ndimage.label(birds_binary_clean > 0)
    print(f'Foreground components (SciPy, default structure): {scipy_count}')
except ImportError:
    print('SciPy is not available, so the comparison with ndimage.label is skipped.')


### 2.3 Analysis questions

Answer after testing:
- Does the number of detected connected components look correct?
- If not, is the issue caused by thresholding, touching objects, noise, holes, or connectivity choice?
- Would `4`-connectivity and `8`-connectivity give the same result here? Why?

### 2.4 Alternative approach with `cv2.findContours`

Tasks from the statement:
1. Read the documentation for `cv2.findContours` and explain in a few words what it returns.
2. Extract external and internal contours from the binary image.
3. Draw them on the color version of `voilier_oies_blanches.jpg`.
4. Display external contours in yellow.
5. Add a visualization constraint so that the inside of connected components appears in cyan.

In [ ]:
voilier_color_path = voilier_path
print(f'Voilier color path: {voilier_color_path}')
voilier_color = cv2.imread(str(voilier_color_path), cv2.IMREAD_COLOR)
if voilier_color is None:
    raise FileNotFoundError(f'Unable to read {voilier_color_path}')

contours, hierarchy = cv2.findContours(birds_binary_clean, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
YELLOW = (0, 255, 255)
CYAN = (255, 255, 0)
contour_overlay = voilier_color.copy()
external_count = 0
internal_count = 0

if hierarchy is not None:
    hierarchy = hierarchy[0]
    for idx, contour in enumerate(contours):
        if hierarchy[idx][3] == -1:
            cv2.drawContours(contour_overlay, contours, idx, CYAN, cv2.FILLED)
            external_count += 1
        else:
            internal_count += 1
    for idx, contour in enumerate(contours):
        if hierarchy[idx][3] == -1:
            cv2.drawContours(contour_overlay, contours, idx, YELLOW, 2)

show_color(contour_overlay, 'Contours on voilier image')
print(f'External contours: {external_count}')
print(f'Internal contours: {internal_count}')


## 3. Real-Time Adaptation to Webcam or AVI Input

Tasks:
1. Adapt your contour-based pipeline to a grayscale webcam stream.
2. Point the camera toward a simple object placed on a uniform background.
3. If no webcam is available, use one of the AVI files mentioned in the statement.
4. Save the segmented sequence.
5. Describe and criticize the observed behavior.

In [ ]:
RUN_VIDEO_DEMOS = False
video_candidates = [
    0,
    DATA_DIR / 'Film_AVI_1.avi',
    DATA_DIR / 'Film_AVI_2.avi',
]

if RUN_VIDEO_DEMOS:
    run_te04_segmentation(video_candidates, OUT_DIR)
else:
    print("Set RUN_VIDEO_DEMOS = True to run TE04 segmentation. The demo tries webcam 0 first, then AVI fallbacks. Press 'q' to quit and 's' to save snapshots.")


## 4. Challenge - Count the White Stripes of a Crosswalk

Statement goal:
- automatically count the number of white stripes in `passagepieton.jpg`

Before coding, propose a processing pipeline on paper or in markdown:
1. preprocessing
2. thresholding / segmentation
3. morphology cleanup
4. connected components or contour filtering
5. counting rule
6. final visualization

In [ ]:
passage_path = DATA_DIR / 'passagepieton.jpg'
print(f'Crosswalk image: {passage_path}')

if not passage_path.exists():
    print('Skipping crosswalk challenge because passagepieton.jpg is not available in ../data/te04/.')
else:
    passage = cv2.imread(str(passage_path), cv2.IMREAD_COLOR)
    if passage is None:
        raise FileNotFoundError(f'Unable to read {passage_path}')

    gray = cv2.cvtColor(passage, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    roi_top = height // 2
    roi_gray = gray[roi_top:, :]
    roi_blurred = cv2.GaussianBlur(roi_gray, (5, 5), 0)
    _, roi_binary = cv2.threshold(roi_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 5))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 3))
    roi_clean = cv2.morphologyEx(roi_binary, cv2.MORPH_CLOSE, kernel_close)
    roi_clean = cv2.morphologyEx(roi_clean, cv2.MORPH_OPEN, kernel_open)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(roi_clean, connectivity=8)
    annotated_passage = passage.copy()
    stripe_count = 0

    for label_idx in range(1, num_labels):
        x, y, box_w, box_h, area = stats[label_idx]
        aspect_ratio = box_w / max(box_h, 1)
        if area < 800:
            continue
        if aspect_ratio < 1.8:
            continue
        if box_h > roi_gray.shape[0] * 0.4:
            continue

        stripe_count += 1
        top_left = (x, y + roi_top)
        bottom_right = (x + box_w, y + box_h + roi_top)
        cv2.rectangle(annotated_passage, top_left, bottom_right, (0, 255, 255), 2)
        cv2.putText(
            annotated_passage,
            str(stripe_count),
            (x, y + roi_top - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2,
            cv2.LINE_AA,
        )

    cv2.putText(
        annotated_passage,
        f'Stripes: {stripe_count}',
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 255),
        2,
        cv2.LINE_AA,
    )

    result_path = OUT_DIR / 'passagepieton_result.png'
    cv2.imwrite(str(result_path), annotated_passage)
    show_grid(
        [passage, roi_binary, roi_clean, annotated_passage],
        ['Original image', 'ROI threshold', 'ROI after morphology', f'Annotated stripes = {stripe_count}'],
        cols=2,
        figsize=(12, 8),
    )
    print(f'Saved {result_path.name}')


### Analysis notes

- The threshold value, the ROI placement, and the geometric filtering are the most fragile parts of the crosswalk pipeline.
- The result is sensitive to illumination changes, shadows, perspective, and any bright road markings that survive the threshold.
- Robustness would improve with a better ROI, perspective rectification, stronger shape constraints, and temporal smoothing on video.


## Final Self-Check

- [x] You implemented every code cell.
- [x] You answered the theory and interpretation prompts.
- [x] You compared several structuring elements.
- [x] You labeled connected components and analyzed the count.
- [x] You tested a contour-based alternative.
- [x] You prepared or implemented the crosswalk challenge pipeline.
- [x] You saved outputs inside `../outputs/te04/`.
